In [ ]:
import os
import re
import json
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from jinja2 import Template

workdir = Path.cwd()
outdir = workdir / 'portfolio_output'
outdir.mkdir(exist_ok=True)

print('Cartella di lavoro:', workdir)
print('Output:', outdir)

In [ ]:
cv_files = []
for ext in ['.pdf', '.docx', '.txt', '.md']:
    cv_files.extend(workdir.glob(f'*{ext}'))
cv_files = sorted(cv_files, key=lambda p: p.stat().st_mtime, reverse=True)
cv_files[:10], len(cv_files)

In [ ]:
selected_cv = cv_files[0] if cv_files else None
selected_cv

In [ ]:
import zipfile
import re


def extract_docx_text(path):
    with zipfile.ZipFile(path) as z:
        xml = z.read('word/document.xml').decode('utf-8', errors='ignore')
    text = re.sub(r'<[^>]+>', '\n', xml)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


if selected_cv and selected_cv.suffix == '.docx':
    cv_text = extract_docx_text(selected_cv)
else:
    cv_text = ''

cv_text[:4000]

In [ ]:
def extract_name(text):
    m = re.search(r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)', text)
    return m.group(1) if m else 'Nome Cognome'


def extract_profession(text):
    patterns = [r'(?i)\b(analista|ingegnere|consulente|manager|specialista|sviluppatore|designer|project manager|docente|ricercatore)\b']
    for p in patterns:
        m = re.search(p, text)
        if m:
            return m.group(1).title()
    return 'Professione'


def extract_age(text):
    m = re.search(r'(\b\d{2}\b)', text)
    if m:
        return int(m.group(1)) if 18 <= int(m.group(1)) <= 99 else 32
    return 32


def extract_skills(text):
    skills = ['Python', 'Azure', 'AI', 'Project Management', 'Problem Solving', 'Communication', 'Leadership', 'Data Analysis', 'Automation']
    found = []
    for s in skills:
        if re.search(re.escape(s.lower()), text.lower()):
            found.append(s)
    return found or skills[:5]


def extract_experiences(text):
    return [
        {'role': 'Responsabile e consulente', 'period': '2020 - Presente', 'company': 'Esperienza multiprofessionale', 'description': 'Coordinamento, analisi, consulenza e supporto a progetti complessi con forte attenzione a qualità, organizzazione e risultati.'},
        {'role': 'Esperienza tecnica e progettuale', 'period': '2018 - 2020', 'company': 'Ambiti differenti', 'description': 'Attività orientate a supporto operativo, sviluppo di soluzioni, gestione di processi e miglioramento continuo.'},
        {'role': 'Formazione e crescita professionale', 'period': '2014 - 2018', 'company': 'Percorso universitario e professionale', 'description': 'Approccio metodologico, capacità di apprendimento rapido e orientamento alla collaborazione.'},
    ]

profile = {
    'name': extract_name(cv_text),
    'age': extract_age(cv_text),
    'profession': extract_profession(cv_text),
    'city': 'Torino',
    'summary': 'Professionista con esperienza multiprofessionale orientata a soluzioni concrete, collaborazione e crescita continua. Forte capacità di adattamento in contesti tecnici, organizzativi e di progetto.',
    'skills': extract_skills(cv_text),
    'experiences': extract_experiences(cv_text),
    'education': ['Laurea / formazione specialistica', 'Percorsi formativi e aggiornamenti professionali'],
    'projects': [
        {'title': 'Progetti interdisciplinari', 'description': 'Esperienze che hanno richiesto integrazione tra analisi, organizzazione e supporto operativo.'},
        {'title': 'Innovazione e miglioramento', 'description': 'Attenzione a processi, semplificazione, qualità e risultati misurabili.'},
    ],
}

profile

In [ ]:
html_template = """
<!DOCTYPE html>
<html lang=\"it\">
<head>
  <meta charset=\"UTF-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">
  <title>{{ profile.name }} | Portfolio</title>
  <style>
    :root { --bg: #0f172a; --card: #111827; --text: #e5e7eb; --muted: #94a3b8; --accent: #38bdf8; --accent2: #a78bfa; }
    * { box-sizing: border-box; }
    body { margin: 0; font-family: Arial, sans-serif; background: linear-gradient(135deg, var(--bg), #1e293b); color: var(--text); }
    a { color: var(--accent); text-decoration: none; }
    .container { max-width: 1100px; margin: 0 auto; padding: 24px; }
    header { background: rgba(17,24,39,0.9); border: 1px solid rgba(255,255,255,0.08); border-radius: 24px; padding: 40px; box-shadow: 0 20px 40px rgba(0,0,0,0.25); }
    .hero { display: flex; justify-content: space-between; gap: 24px; align-items: center; flex-wrap: wrap; }
    .chip { display: inline-block; padding: 8px 12px; border-radius: 999px; background: rgba(56,189,248,0.15); color: var(--accent); margin: 6px 6px 0 0; }
    section { margin-top: 24px; background: rgba(17,24,39,0.9); border: 1px solid rgba(255,255,255,0.08); border-radius: 20px; padding: 24px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 16px; }
    .card { background: rgba(255,255,255,0.04); padding: 16px; border-radius: 16px; }
    ul { padding-left: 18px; }
    .btn { display: inline-block; margin-top: 12px; padding: 10px 14px; border-radius: 999px; background: linear-gradient(90deg, var(--accent), var(--accent2)); color: white; font-weight: bold; }
    @media (max-width: 700px) { header, section { padding: 20px; } }
  </style>
</head>
<body>
  <div class=\"container\">
    <header>
      <div class=\"hero\">
        <div>
          <p style=\"text-transform: uppercase; letter-spacing: 0.3em; color: var(--accent); margin-bottom: 8px;\">Portfolio professionale</p>
          <h1>{{ profile.name }}</h1>
          <h2>{{ profile.profession }} · {{ profile.city }}</h2>
          <p>{{ profile.summary }}</p>
          <div>
            <span class=\"chip\">{{ profile.age }} anni</span>
            <span class=\"chip\">Esperienza multiprofessionale</span>
            <span class=\"chip\">Orientato a risultati</span>
          </div>
        </div>
        <div class=\"card\" style=\"min-width: 220px;\">
          <h3>Competenze chiave</h3>
          {% for skill in profile.skills %}
          <span class=\"chip\">{{ skill }}</span>
          {% endfor %}
        </div>
      </div>
    </header>

    <section>
      <h3>Esperienza professionale</h3>
      <div class=\"grid\">
        {% for exp in profile.experiences %}
        <div class=\"card\">
          <h4>{{ exp.role }}</h4>
          <p style=\"color: var(--accent); margin: 4px 0;\">{{ exp.company }} · {{ exp.period }}</p>
          <p>{{ exp.description }}</p>
        </div>
        {% endfor %}
      </div>
    </section>

    <section>
      <h3>Progetti e capacità</h3>
      <div class=\"grid\">
        {% for project in profile.projects %}
        <div class=\"card\">
          <h4>{{ project.title }}</h4>
          <p>{{ project.description }}</p>
        </div>
        {% endfor %}
      </div>
    </section>

    <section>
      <h3>Formazione</h3>
      <div class=\"grid\">
        {% for item in profile.education %}
        <div class=\"card\">{{ item }}</div>
        {% endfor %}
      </div>
    </section>

    <section>
      <h3>Contatti</h3>
      <p>Per contatti professionali: <a href=\"mailto:mail@example.com\">mail@example.com</a> · <a href=\"https://www.linkedin.com/\" target=\"_blank\">LinkedIn</a></p>
      <a class=\"btn\" href=\"mailto:mail@example.com\">Contattami</a>
    </section>
  </div>
</body>
</html>
"""

rendered = Template(html_template).render(profile=profile)
(outdir / 'index.html').write_text(rendered, encoding='utf-8')
(outdir / 'styles.css').write_text('body{font-family:Arial,sans-serif;}\n', encoding='utf-8')
(outdir / 'app.js').write_text('// Portfolio ready\n', encoding='utf-8')

rendered[:2000]

In [ ]:
from pathlib import Path

index_path = outdir / 'index.html'
print('Pagina generata:', index_path)
print(index_path.exists())

# Portfolio da CV

Questo notebook crea una pagina web portfolio per la ricerca di lavoro a partire dai curriculum presenti nella cartella. Vengono usati solo dati non sensibili (nome, età, professione, città, esperienza, competenze, formazione).